In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import pandas as pd
import altair as alt
sys.path.append("..")
from budget.grist import get_records, records_to_df

In [13]:
# lecture des données sur Grist
df_conso = records_to_df(get_records("Consommations"))
df_periode = records_to_df(get_records("Periode"))
df_conso_periode = df_conso.merge(df_periode, left_on="periode", right_on="id", suffixes=("", "_periode"))
#lecture des données csv
df_conso_previsionnelle = pd.read_csv("../budget/conso_previsionnelle.csv", sep=",")
df_sponsors = pd.read_csv("../budget/sponsors.csv", sep=",")
df_recettes = pd.read_csv("../budget/recettes.csv", sep=",")


In [14]:
# mise en forme (annee, statut des financements)
df_conso_periode['annee'] = pd.to_datetime(df_conso_periode['mois'], unit='s', utc=True).dt.year
df_conso_periode['statut'] = 'consommée'
# ajout de la ligne prévisionnelle
df_conso_periode = pd.concat([df_conso_periode, pd.DataFrame(df_conso_previsionnelle)], ignore_index=True)
df_conso_annuelle = (
    df_conso_periode
    .set_index('annee')['montant_ttc']
    .groupby('annee')
    .sum()
    .reset_index()
)

In [15]:
chart_conso = alt.Chart(df_conso_annuelle).mark_bar(size=40).encode(
    x=alt.X('annee:O', title='Année'),
    y=alt.Y('montant_ttc:Q', title='Montant TTC (€)', scale=alt.Scale(domain=[0, 400000])),
    tooltip=['annee:O', 'montant_ttc:Q'],
).properties(
    title='Consommations annuelles (historique + prévisionnel)',
    width=300)
chart_conso

alt.Chart(...)

In [16]:
chart_sponsor = alt.Chart(df_sponsors).mark_bar(size=40).encode(
    x=alt.X('annee:O', title='Année'),
    y=alt.Y('montant_ttc:Q', title='Montant TTC (€)', scale=alt.Scale(domain=[0, 400000])),
    color=alt.Color('sponsor:N', title='Sponsor', legend=alt.Legend(orient='top-left')),
    tooltip=['annee:O', 'sponsor:N', 'montant_ttc:Q'],
).properties(
    title='Sponsoring annuel',
    width=300)
chart_sponsor


alt.Chart(...)

In [17]:
chart_recette = alt.Chart(df_recettes).mark_bar(size=40).encode(
    x=alt.X('annee:O', title='Année'),
    y=alt.Y('montant_ttc:Q', title='Montant TTC (€)', scale=alt.Scale(domain=[0, 400000])),
    color=alt.Color('client:N', title='Client', legend=alt.Legend(orient='top-left')),
    tooltip=['annee:O', 'client:N', 'montant_ttc:Q'],
).properties(
    title='Recettes annuelles par client',
    width=300)
chart_recette


alt.Chart(...)

In [9]:
df_conso_annuelle

,annee,montant_ttc
0,2024,15552.0
1,2025,189489.6
2,2026,386048.0
3,2027,400000.0
